# Final Battery SOH Evaluation

This notebook audits the repository dataset and displays results from `backend/ml_models/train_best_battery_model.py`. The evaluation separates unseen-battery generalization from known-battery rolling one-step forecasting. The under-1.5 percentage-point claim applies only to the latter setup. `rul` is excluded because it encodes the final observed cycle. The target SOH is capacity divided by the battery's first recorded capacity.

In [1]:
from pathlib import Path
import json
import pandas as pd

repo = Path.cwd()
if not (repo / 'data' / 'battery_cycle_level_dataset_CLEAN_FINAL.csv').exists():
    repo = repo.parent
dataset_path = repo / 'data' / 'battery_cycle_level_dataset_CLEAN_FINAL.csv'
df = pd.read_csv(dataset_path).sort_values(['battery_id', 'cycle']).reset_index(drop=True)
print('Dataset:', dataset_path)
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Dtypes and unique values:')
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'unique': df.nunique(), 'missing': df.isna().sum()}))
print('Battery IDs:', sorted(df['battery_id'].unique().tolist()))
print('Cycles per battery:')
display(df.groupby('battery_id').agg(cycles=('cycle', 'size'), first_cycle=('cycle', 'min'), last_cycle=('cycle', 'max')))
print('Numeric distributions:')
display(df.select_dtypes('number').describe(percentiles=[.05, .25, .5, .75, .95]).T)
print('Numeric correlations:')
display(df.select_dtypes('number').corr())
print('Duplicate rows:', int(df.duplicated().sum()))
initial_capacity = df.groupby('battery_id')['capacity'].transform('first')
print('Max abs error in SOH - capacity/initial_capacity:', float((df['soh'] - df['capacity'] / initial_capacity).abs().max()))
future_rul = df.groupby('battery_id')['cycle'].transform('max') - df['cycle']
print('Rows where RUL equals final_cycle-cycle:', int(df['rul'].eq(future_rul).sum()), '/', len(df))

Dataset: c:\Users\supri\Downloads\AutoCircle-AI-Productionnn\AutoCircle-AI\data\battery_cycle_level_dataset_CLEAN_FINAL.csv
Shape: (1415, 7)
Columns: ['battery_id', 'cycle', 'voltage', 'temperature', 'capacity', 'soh', 'rul']
Dtypes and unique values:


,dtype,unique,missing
battery_id,object,34,0
cycle,int64,168,0
voltage,float64,1415,0
temperature,float64,1415,0
capacity,float64,1415,0
soh,float64,1382,0
rul,int64,168,0


Battery IDs: ['B0005', 'B0006', 'B0007', 'B0018', 'B0025', 'B0026', 'B0027', 'B0028', 'B0029', 'B0030', 'B0031', 'B0032', 'B0033', 'B0034', 'B0036', 'B0038', 'B0039', 'B0040', 'B0041', 'B0042', 'B0043', 'B0044', 'B0045', 'B0046', 'B0047', 'B0048', 'B0049', 'B0050', 'B0051', 'B0052', 'B0053', 'B0054', 'B0055', 'B0056']
Cycles per battery:


,cycles,first_cycle,last_cycle
battery_id,,,
B0005,168,1,168
B0006,155,1,155
B0007,168,1,168
B0018,132,1,132
B0025,5,1,12
B0026,18,1,25
B0027,14,1,26
B0028,27,1,28
B0029,32,1,40


Numeric distributions:


,count,mean,std,min,5%,25%,50%,75%,95%,max
cycle,1415.0,55.389399,42.728497,1.000000,4.000000,21.000000,43.000000,87.000000,141.000000,168.000000
voltage,1415.0,3.420612,0.166460,2.530010,3.075679,3.387078,3.452596,3.507450,3.564381,3.753027
temperature,1415.0,26.589773,13.196045,7.684550,7.986292,9.555186,31.139259,32.753400,53.694017,54.997372
capacity,1415.0,1.501191,0.316995,0.520105,0.856953,1.321640,1.504115,1.709022,1.976892,2.498996
soh,1415.0,0.849014,0.119219,0.604449,0.649861,0.750777,0.850350,0.961174,1.020127,1.048527
rul,1415.0,54.175972,43.709204,0.000000,2.000000,16.500000,43.000000,86.000000,140.000000,167.000000


Numeric correlations:


,cycle,voltage,temperature,capacity,soh,rul
cycle,1.000000,0.036271,0.030307,-0.256151,-0.622161,-0.355340
voltage,0.036271,1.000000,0.249777,0.466416,0.257508,0.368511
temperature,0.030307,0.249777,1.000000,0.536293,0.418583,0.174537
capacity,-0.256151,0.466416,0.536293,1.000000,0.643486,0.448028
soh,-0.622161,0.257508,0.418583,0.643486,1.000000,0.332720
rul,-0.355340,0.368511,0.174537,0.448028,0.332720,1.000000


Duplicate rows: 0
Max abs error in SOH - capacity/initial_capacity: 4.440892098500626e-16
Rows where RUL equals final_cycle-cycle: 1415 / 1415


In [9]:
metrics_path = repo / 'results' / 'final_battery' / 'final_battery_metrics.json'
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print('UNSEEN BATTERIES')
print(json.dumps(metrics['unseen_battery'], indent=2))
print('KNOWN BATTERY / FUTURE CYCLES')
print(json.dumps(metrics['known_battery_future_cycles'], indent=2))
print('Under 1.5 percentage points for rolling forecast:', metrics['claim_under_1_5_percentage_points'])

UNSEEN BATTERIES
{
  "protocol": "GroupShuffleSplit by battery ID (train/validation/test); test rows exclude cycle 1; features cycle, voltage, temperature, and first-cycle capacity; current capacity and RUL excluded.",
  "validation_winner": {
    "formulation": "direct_soh",
    "model": "hist_gradient_boosting",
    "mae_fraction": 0.03612092179087743,
    "mae_percentage_points": 3.6120921790877434,
    "rmse_fraction": 0.04676502042145233,
    "rmse_percentage_points": 4.676502042145233,
    "r2": 0.7840068171548884,
    "rows": 460
  },
  "test": {
    "mae_fraction": 0.048500748124080464,
    "mae_percentage_points": 4.850074812408047,
    "rmse_fraction": 0.0845809983876961,
    "rmse_percentage_points": 8.458099838769611,
    "r2": 0.5443436257990777,
    "rows": 358
  },
  "test_batteries": [
    "B0005",
    "B0029",
    "B0030",
    "B0032",
    "B0042",
    "B0050"
  ],
  "test_rows": 358,
  "group_kfold_5_fold": {
    "protocol": "5-fold GroupKFold; each battery is held ou

In [10]:
from pathlib import Path
artifact_path = repo / 'backend' / 'ml_models' / 'artifacts' / 'final_battery_model.joblib'
print('Saved estimator:', artifact_path, 'exists:', artifact_path.exists())
leaderboard_path = repo / 'results' / 'final_battery' / 'future_cycle_validation_leaderboard.csv'
per_battery_path = repo / 'results' / 'final_battery' / 'future_cycle_per_battery_errors.csv'
display(pd.read_csv(leaderboard_path).head(20))
display(pd.read_csv(per_battery_path).sort_values('MAE_percentage_points', ascending=False))
print('Per-battery actual/predicted figure:', (repo / 'results' / 'final_battery' / 'figures' / 'future_forecast_by_battery.svg').exists())

Saved estimator: c:\Users\supri\Downloads\AutoCircle-AI-Productionnn\AutoCircle-AI\backend\ml_models\artifacts\final_battery_model.joblib exists: True


,formulation,model,mae_fraction,mae_percentage_points,rmse_fraction,rmse_percentage_points,r2,rows
0,degradation,gradient_boosting,0.006031,0.603129,0.011655,1.165547,0.989307,270
1,delta_soh,gradient_boosting,0.006081,0.608091,0.011765,1.176460,0.989106,270
2,delta_soh,xgboost_3,0.006302,0.630232,0.011300,1.129960,0.989950,270
3,degradation,xgboost_3,0.006302,0.630232,0.011300,1.129960,0.989950,270
4,delta_soh,xgboost_5,0.006303,0.630302,0.011301,1.130052,0.989949,270
5,degradation,xgboost_5,0.006303,0.630302,0.011301,1.130052,0.989949,270
6,degradation,xgboost_6,0.006304,0.630423,0.011302,1.130216,0.989946,270
7,delta_soh,xgboost_6,0.006304,0.630423,0.011302,1.130216,0.989946,270
8,degradation,xgboost_1,0.006417,0.641729,0.011409,1.140909,0.989755,270
9,delta_soh,xgboost_1,0.006417,0.641729,0.011409,1.140909,0.989755,270


,battery_id,MAE_percentage_points,RMSE_percentage_points,R2,number_of_cycles
4,B0025,8.063187,8.063187,NaN,1
21,B0049,6.310074,6.313025,-4.192648,2
5,B0026,4.246775,5.645178,-2.477267,4
7,B0028,2.570377,2.969280,-1.699541,6
6,B0027,1.717901,2.347385,-0.715420,3
18,B0046,1.082728,2.178585,-0.530138,14
20,B0048,0.956185,1.749986,-0.468848,14
17,B0045,0.881355,1.128882,-0.188150,14
12,B0039,0.844950,0.859621,0.461917,3
22,B0053,0.831659,0.990899,0.193281,9


Per-battery actual/predicted figure: True


## Re-run the benchmark

The final cell retrains from the repository CSV, repeats the validation-based model/formulation comparison, evaluates the untouched chronological test segments, and rewrites the model, metrics, predictions, per-battery error table, SVG, and report. The split is deterministic with seed 42.

In [8]:
import runpy
_ = runpy.run_path(str(repo / 'backend' / 'ml_models' / 'train_best_battery_model.py'), run_name='__main__')


{
  "dataset_rows": 1415,
  "battery_count": 34,
  "soh_capacity_formula_max_error": 4.440892098500626e-16,
  "rul_future_leakage_rows": 1415,
  "unseen_battery_test": {
    "mae_fraction": 0.048500748124080464,
    "mae_percentage_points": 4.850074812408047,
    "rmse_fraction": 0.0845809983876961,
    "rmse_percentage_points": 8.458099838769611,
    "r2": 0.5443436257990777,
    "rows": 358
  },
  "unseen_battery_validation_winner": {
    "formulation": "direct_soh",
    "model": "hist_gradient_boosting",
    "mae_fraction": 0.03612092179087743,
    "mae_percentage_points": 3.6120921790877434,
    "rmse_fraction": 0.04676502042145233,
    "rmse_percentage_points": 4.676502042145233,
    "r2": 0.7840068171548884,
    "rows": 460
  },
  "future_cycle_test": {
    "mae_fraction": 0.006951996067044132,
    "mae_percentage_points": 0.6951996067044132,
    "rmse_fraction": 0.014587356344485467,
    "rmse_percentage_points": 1.4587356344485467,
    "r2": 0.9837910628526646,
    "rows": 288
